In [1]:
from autogen import Agent, AssistantAgent, UserProxyAgent
from pathlib import Path
from autogen.coding import LocalCommandLineCodeExecutor
import requests

In [2]:
# Setup your actual server IP and port (Ollama uses port 11434 by default)
ollama_host = "http://192.168.1.10:11434"

# Endpoint to list models
url = f"{ollama_host}/api/tags"

# Get the list of installed models
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    print("Installed Models:")
    for model in data.get("models", []):
        print(f"- {model['name']}")
else:
    print(f"Failed to fetch models: {response.status_code}")
    print(response.text)


Installed Models:
- qwen3:14b
- huihui_ai/qwen3-abliterated:14b
- qwen3:8b
- huihui_ai/qwen3-abliterated:8b
- qwen3:32b
- gemma3:4b
- gemma3:4b-it-qat
- mxbai-embed-large:latest
- nomic-embed-text:latest
- phi4-reasoning:plus
- phi4-mini-reasoning:latest
- phi4-reasoning:latest
- phi4:latest
- gemma3:12b
- deepcoder:14b
- deepcoder:1.5b
- llama3.3:latest
- phi:latest
- qwen2.5:14b-instruct
- gemma3:27b
- deepseek-r1:70b
- deepseek-r1:32b
- deepseek-r1:14b
- deepseek-r1:8b
- qwen2:7b
- qwen2.5-coder:32b-instruct-q5_0
- qwen2.5-coder:14b-instruct-q5_0
- llama3.1:70b
- llama3.1:8b
- gemma2:27b
- qwen2.5:72b
- qwen2.5:32b
- qwen2.5:14b
- llama3.2:latest
- qwen2.5:1.5b
- deepseek-coder:1.3b
- qwen2.5-coder:0.5b
- qwen2.5:7b
- qwen2.5:0.5b
- qwen2.5-coder:14b
- qwen2.5-coder:7b
- qwen2.5-coder:3b
- llama3.2-vision:latest
- qwen2.5-coder:1.5b
- krith/qwen2.5-72b-instruct:IQ1_M
- qwen2:72b-instruct
- mistral:latest
- qwen2:72b
- llava:34b-v1.6
- llava-llama3:8b-v1.1-fp16
- gemma2:9b
- llava:13

In [3]:
config_list = [
    {
        # Let's choose the qwen3(8b) model (model names must match Ollama exactly)
        "model": "qwen3:8b",
        # We specify the API Type as 'ollama' so it uses the Ollama client class
        "api_type": "ollama",
        "stream": False,
        "client_host": "http://192.168.1.10:11434",
    }
]

#  Two-Agent Coding Example

In [4]:
# Setting up the code executor
workdir = Path("coding")
workdir.mkdir(exist_ok=True)
code_executor = LocalCommandLineCodeExecutor(work_dir=workdir)

# Setting up the agents

# The UserProxyAgent will execute the code that the AssistantAgent provides
user_proxy_agent = UserProxyAgent(
    name="User",
    code_execution_config={"executor": code_executor},
    is_termination_msg=lambda msg: "FINISH" in msg.get("content"),
)

system_message = """You are a helpful AI assistant who writes code and the user
executes it. Solve tasks using your python coding skills.
In the following cases, suggest python code (in a python coding block) for the
user to execute. When using code, you must indicate the script type in the code block.
You only need to create one working sample.
Do not suggest incomplete code which requires users to modify it.
Don't use a code block if it's not intended to be executed by the user. Don't
include multiple code blocks in one response. Do not ask users to copy and
paste the result. Instead, use 'print' function for the output when relevant.
Check the execution result returned by the user.

If the result indicates there is an error, fix the error.

IMPORTANT: If it has executed successfully, ONLY output 'FINISH'."""

# The AssistantAgent, using the Ollama config, will take the coding request and return code
assistant_agent = AssistantAgent(
    name="Ollama Assistant",
    system_message=system_message,
    llm_config={"config_list": config_list},
)

In [5]:
# Start the chat, with the UserProxyAgent asking the AssistantAgent the message
chat_result = user_proxy_agent.initiate_chat(
    assistant_agent,
    message="Provide code to count the number of prime numbers from 1 to 10000.",
)

User (to Ollama Assistant):

Provide code to count the number of prime numbers from 1 to 10000.

--------------------------------------------------------------------------------
Ollama Assistant (to User):

<think>
Okay, I need to write a Python script that counts the number of prime numbers between 1 and 10,000. Let me think about how to approach this.

First, I remember that a prime number is a number greater than 1 that has no divisors other than 1 and itself. So, for each number in the range, I need to check if it's prime.

Wait, but checking every number from 2 to 10,000 might be time-consuming. Maybe there's an efficient way. Oh right, the Sieve of Eratosthenes algorithm is a good method for finding all primes up to a certain limit. That's probably faster than checking each number individually.

Let me outline the steps for the sieve. Start by creating a list of boolean values initialized to True, representing numbers from 0 to n (10,000). Then set index 0 and 1 to False since th

# Three Agents Coding Example

In [6]:
# Configuration for each model
base_url = "http://192.168.1.10:11434"
model_names = ["qwen3:8b", "gemma3:4b", "deepcoder:1.5b"]

In [7]:
# Shared system message
system_message = """You are a helpful AI assistant who writes code and the user
executes it. Solve tasks using your python coding skills.
In the following cases, suggest python code (in a python coding block) for the
user to execute. When using code, you must indicate the script type in the code block.
You only need to create one working sample.
Do not suggest incomplete code which requires users to modify it.
Don't use a code block if it's not intended to be executed by the user. Don't
include multiple code blocks in one response. Do not ask users to copy and
paste the result. Instead, use 'print' function for the output when relevant.
Check the execution result returned by the user.

If the result indicates there is an error, fix the error.

IMPORTANT: If it has executed successfully, ONLY output 'FINISH'."""


In [8]:
# Create working directory and executor
workdir = Path("coding")
workdir.mkdir(exist_ok=True)
code_executor = LocalCommandLineCodeExecutor(work_dir=workdir)

In [9]:
# Create UserProxyAgent
user_proxy_agent = UserProxyAgent(
    name="User",
    code_execution_config={"executor": code_executor},
    is_termination_msg=lambda msg: "FINISH" in msg.get("content"),
)

In [10]:
# Create AssistantAgents for each model
assistant_agents = []
for model in model_names:
   agent = AssistantAgent(
       name=f"Assistant_{model.replace(':', '_')}",
       system_message=system_message,
       llm_config={
           "config_list": [
               {
                   "model": model,
                   "api_type": "ollama",
                   "stream": False,
                   "client_host": base_url,
               }
            ]
        },
   )
   assistant_agents.append(agent)

In [11]:
# Run the task with each assistant in sequence
task = "Provide code to count the number of prime numbers from 1 to 10000."

for assistant in assistant_agents:
    print(f"\n=== Running with {assistant.name} ===")
    result = user_proxy_agent.initiate_chat(
        assistant,
        message=task
    )


=== Running with Assistant_qwen3_8b ===
User (to Assistant_qwen3_8b):

Provide code to count the number of prime numbers from 1 to 10000.

--------------------------------------------------------------------------------
Assistant_qwen3_8b (to User):

<think>
Okay, I need to write a Python script that counts the number of prime numbers between 1 and 10,000. Let me think about how to approach this.

First, I remember that a prime number is a number greater than 1 that has no divisors other than 1 and itself. So, for each number in the range, I need to check if it's prime.

Wait, but checking every number from 2 to 10,000 might be time-consuming. Maybe there's a more efficient way. Oh right, the Sieve of Eratosthenes algorithm is a good method for finding all primes up to a given limit. That could be faster than checking each number individually.

Let me outline the steps for the sieve. Start by creating a list of boolean values representing whether each number is prime. Initialize all e